# Proyecto Final Econometría II
## Gabriel Mejía & David Romero

## 1. Imports

In [11]:
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.stats import skew
from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV, ElasticNetCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

from feature_engine.imputation import CategoricalImputer, MeanMedianImputer, AddMissingIndicator
from feature_engine.encoding import CountFrequencyEncoder

import myPreprocessors as mypp

SEED = 2026
np.random.seed(SEED)

def fix_r_colnames(df):
    rename = {c: re.sub(r'^X(\d)', r'\1', c) for c in df.columns if re.match(r'^X\d', c)}
    return df.rename(columns=rename)

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

train_raw = fix_r_colnames(pd.read_csv('data_train_v3.csv'))
test_raw  = fix_r_colnames(pd.read_csv('test_v3.csv'))
test_ids = test_raw['Id'].copy()
print(f'Train: {train_raw.shape}  Test: {test_raw.shape}')

Train: (880, 81)  Test: (220, 80)


## 2. Pipeline A — frequency-encoding + Ridge/GradientBoosting (feature_engine)

Mismo pipeline validado en la versión que dio 26,485 en Kaggle.

In [12]:
y_a = train_raw['SalePrice'].copy()
Xa = train_raw.drop(columns=['Id', 'SalePrice'])
Xa_test = test_raw.drop(columns=['Id'])

mask_outlier_a = (Xa['GrLivArea'] > 4000) & (y_a < 300_000)
Xa = Xa[~mask_outlier_a].reset_index(drop=True)
y_a = y_a[~mask_outlier_a].reset_index(drop=True)

def add_features_a(df):
    df = df.copy()
    df['TotalSF']      = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath']    = (df['FullBath'] + 0.5 * df['HalfBath'] +
                           df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath'])
    df['HouseAge']     = df['YrSold'] - df['YearBuilt']
    df['RemodAge']     = df['YrSold'] - df['YearRemodAdd']
    df['HasPool']      = (df['PoolArea'] > 0).astype(int)
    df['HasGarage']    = (df['GarageArea'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
    df['Has2ndFlr']    = (df['2ndFlrSF'] > 0).astype(int)
    df['PorchSF']      = (df['OpenPorchSF'] + df['EnclosedPorch'] +
                           df['3SsnPorch'] + df['ScreenPorch'])
    return df

Xa = add_features_a(Xa)
Xa_test = add_features_a(Xa_test)

CAT_NA_MISSING = ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
                   'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageType',
                   'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence',
                   'MiscFeature']
CAT_NA_FREQUENT = ['MSZoning', 'Exterior1st', 'Exterior2nd', 'Electrical',
                    'KitchenQual', 'SaleType', 'Functional', 'Utilities']
NUM_NA = ['LotFrontage', 'MasVnrArea', 'GarageYrBlt']
TEMPORAL_VARS = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']
REF_VAR = 'YrSold'
QUAL_VARS = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
QUAL_MAPPINGS = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5, 'Missing': 0, 'NA': 0}
FREQ_ENCODE_VARS = ['MSZoning', 'LotShape', 'LandContour', 'LotConfig', 'LandSlope',
                     'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
                     'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
                     'Foundation', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating',
                     'CentralAir', 'Electrical', 'Functional', 'GarageType', 'GarageFinish',
                     'PavedDrive', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition',
                     'Street', 'Alley', 'Utilities']
LOG_VARS = ['LotFrontage', 'LotArea', 'GrLivArea', '1stFlrSF', 'TotalSF']

class Log1pTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, variables):
        self.variables = variables
    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self
    def transform(self, X):
        X = X.copy()
        for var in self.variables:
            X[var] = np.log1p(X[var].clip(lower=0))
        return X

class FillRemainingNA(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self
    def transform(self, X):
        return X.fillna(0)

preprocessing_steps_a = [
    ('missing_imputation', CategoricalImputer(imputation_method='missing', variables=CAT_NA_MISSING)),
    ('frequent_imputation', CategoricalImputer(imputation_method='frequent', variables=CAT_NA_FREQUENT)),
    ('missing_indicator', AddMissingIndicator(variables=NUM_NA)),
    ('median_imputation', MeanMedianImputer(imputation_method='median', variables=NUM_NA)),
    ('temporal_variables', mypp.TremporalVariableTransformer(variables=TEMPORAL_VARS, reference_variable=REF_VAR)),
    ('quality_mapper', mypp.Mapper(variables=QUAL_VARS, mappings=QUAL_MAPPINGS)),
    ('categorical_encoder', CountFrequencyEncoder(variables=FREQ_ENCODE_VARS, encoding_method='count')),
    ('unseen_category_fix', FillRemainingNA()),
    ('log_transform', Log1pTransformer(variables=LOG_VARS)),
]

def make_pipeline_a(estimator, scale=True):
    steps = list(preprocessing_steps_a)
    if scale:
        steps.append(('scaler', MinMaxScaler()))
    steps.append(('model', estimator))
    return Pipeline(steps)

y_a_log = np.log1p(y_a)
alphas_ridge = np.logspace(-2, 3, 30)
print('Pipeline A listo. Xa:', Xa.shape)

Pipeline A listo. Xa: (879, 88)


## 3. Pipeline B — one-hot encoding completo + ElasticNet/SVR/XGBoost

In [13]:
y_full_b = train_raw['SalePrice'].copy()
n_train_full_b = len(train_raw)
mask_outlier_b = (train_raw['GrLivArea'] > 4000) & (train_raw['SalePrice'] < 300_000)

full_b = pd.concat([train_raw.drop(columns=['SalePrice']), test_raw], ignore_index=True)

none_vars = ['Alley','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
             'FireplaceQu','GarageType','GarageFinish','GarageQual','GarageCond','PoolQC',
             'Fence','MiscFeature','MasVnrType']
for v in none_vars:
    if v in full_b.columns:
        full_b[v] = full_b[v].fillna('None').replace('', 'None')

zero_vars = ['GarageYrBlt','MasVnrArea','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF',
             'BsmtFullBath','BsmtHalfBath','GarageCars','GarageArea']
for v in zero_vars:
    if v in full_b.columns:
        full_b[v] = full_b[v].fillna(0)

for nb in full_b['Neighborhood'].unique():
    mask = full_b['Neighborhood'] == nb
    med = full_b.loc[mask, 'LotFrontage'].median()
    if pd.isna(med):
        med = full_b['LotFrontage'].median()
    full_b.loc[mask & full_b['LotFrontage'].isna(), 'LotFrontage'] = med

for c in full_b.columns:
    if not pd.api.types.is_numeric_dtype(full_b[c]):
        if full_b[c].isna().any() or (full_b[c].astype(str) == '').any():
            mode_val = full_b[c].mode(dropna=True)
            mode_val = mode_val.iloc[0] if len(mode_val) else 'Missing'
            full_b[c] = full_b[c].fillna(mode_val)
    else:
        if full_b[c].isna().any():
            full_b[c] = full_b[c].fillna(full_b[c].median())

qmap = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
qual_vars = ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC','KitchenQual',
             'FireplaceQu','GarageQual','GarageCond','PoolQC']
for v in qual_vars:
    if v in full_b.columns:
        full_b[v + '_o'] = full_b[v].map(qmap).fillna(0)

full_b['TotalSF']      = full_b['TotalBsmtSF'] + full_b['1stFlrSF'] + full_b['2ndFlrSF']
full_b['TotalBath']    = full_b['FullBath'] + 0.5*full_b['HalfBath'] + full_b['BsmtFullBath'] + 0.5*full_b['BsmtHalfBath']
full_b['HouseAge']     = full_b['YrSold'] - full_b['YearBuilt']
full_b['RemodAge']     = full_b['YrSold'] - full_b['YearRemodAdd']
full_b['GarageAge']    = np.where(full_b['GarageYrBlt'] == 0, 0, full_b['YrSold'] - full_b['GarageYrBlt'])
full_b['HasGarage']    = (full_b['GarageArea'] > 0).astype(int)
full_b['HasBsmt']      = (full_b['TotalBsmtSF'] > 0).astype(int)
full_b['HasFireplace'] = (full_b['Fireplaces'] > 0).astype(int)
full_b['HasPool']      = (full_b['PoolArea'] > 0).astype(int)
full_b['TotalPorch']   = (full_b['OpenPorchSF'] + full_b['EnclosedPorch'] + full_b['3SsnPorch'] +
                           full_b['ScreenPorch'] + full_b['WoodDeckSF'])
full_b['QualSF']       = full_b['OverallQual'] * full_b['GrLivArea']
full_b['QualTotalSF']  = full_b['OverallQual'] * full_b['TotalSF']
full_b['QualBath']     = full_b['OverallQual'] * full_b['TotalBath']
full_b['AgeQual']      = full_b['OverallQual'] * np.clip(100 - full_b['HouseAge'], 0, None)
full_b['MSSubClass']   = full_b['MSSubClass'].astype(str)

num_cols_b = [c for c in full_b.columns if pd.api.types.is_numeric_dtype(full_b[c]) and c != 'Id']
for c in num_cols_b:
    s = skew(full_b[c].astype(float))
    if abs(s) > 0.75:
        mn = full_b[c].min()
        full_b[c] = np.log1p(full_b[c] - mn) if mn < 0 else np.log1p(full_b[c])

cat_cols_b = [c for c in full_b.columns if not pd.api.types.is_numeric_dtype(full_b[c]) and c != 'Id']
full_b_dummies = pd.get_dummies(full_b.drop(columns=['Id']), columns=cat_cols_b, drop_first=True)

Xb_all = full_b_dummies.iloc[:n_train_full_b].reset_index(drop=True)
Xb_test = full_b_dummies.iloc[n_train_full_b:].reset_index(drop=True)

Xb_anchor = Xb_all[mask_outlier_b.values].reset_index(drop=True)
anchor_price = y_full_b[mask_outlier_b.values].median()

Xb = Xb_all[~mask_outlier_b.values].reset_index(drop=True)
y_b = y_full_b[~mask_outlier_b.values].reset_index(drop=True)
y_b_log = np.log1p(y_b)

print('Pipeline B listo. Xb:', Xb.shape, ' Xb_test:', Xb_test.shape)
print('Precio real del ancla (Id 524):', anchor_price)
# Nota: Xa y Xb deben tener las mismas filas en el mismo orden (mismo train_raw, mismo outlier)
assert len(Xa) == len(Xb), 'Pipeline A y B deben tener el mismo numero de filas de entrenamiento'

Pipeline B listo. Xb: (879, 291)  Xb_test: (220, 291)
Precio real del ancla (Id 524): 184750.0


## 4. Modelos base de cada pipeline

In [14]:
def fit_en(xtr, ytr, xte):
    model = ElasticNetCV(l1_ratio=[.1, .5, .9, 1], alphas=np.logspace(-4, 0, 20),
                          max_iter=20000, cv=5, random_state=SEED, n_jobs=-1)
    model.fit(xtr, ytr)
    return np.expm1(model.predict(xte))

def fit_svr(xtr, ytr, xte):
    scaler = RobustScaler().fit(xtr)
    xtr_s = scaler.transform(xtr)
    xte_s = scaler.transform(xte)
    model = SVR(kernel='rbf', C=20, gamma=6e-4, epsilon=0.01)
    model.fit(xtr_s, ytr)
    return np.expm1(model.predict(xte_s))

GB_SEEDS = [11, 22, 33, 44, 55]

def fit_gb_bagged(xtr_df, ytr, xte_df):
    preds = []
    for s in GB_SEEDS:
        est = GradientBoostingRegressor(n_estimators=800, learning_rate=0.02, max_depth=3,
                                         subsample=0.45, random_state=s)
        pipe = make_pipeline_a(est, scale=False)
        pipe.fit(xtr_df, ytr)
        preds.append(pipe.predict(xte_df))
    return np.expm1(np.mean(preds, axis=0))

def fit_ridge_a(xtr_df, ytr, xte_df):
    pipe = make_pipeline_a(RidgeCV(alphas=alphas_ridge), scale=True)
    pipe.fit(xtr_df, ytr)
    return np.expm1(pipe.predict(xte_df))

## 5. Validación cruzada

In [15]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_ridge_a = np.zeros(len(Xa))
oof_gb_a    = np.zeros(len(Xa))
oof_en_b    = np.zeros(len(Xb))
oof_svr_b   = np.zeros(len(Xb))

for i, (tr, va) in enumerate(kf.split(Xa)):
    oof_ridge_a[va] = fit_ridge_a(Xa.iloc[tr], y_a_log.iloc[tr], Xa.iloc[va])
    oof_gb_a[va]    = fit_gb_bagged(Xa.iloc[tr], y_a_log.iloc[tr], Xa.iloc[va])
    oof_en_b[va]    = fit_en(Xb.iloc[tr], y_b_log.iloc[tr], Xb.iloc[va])
    oof_svr_b[va]   = fit_svr(Xb.iloc[tr], y_b_log.iloc[tr], Xb.iloc[va])
    print(f'Fold {i+1}/{N_FOLDS} listo')

pipeline_a_oof = 0.41 * oof_ridge_a + 0.59 * oof_gb_a
pipeline_b_oof = 0.145 * oof_en_b + 0.855 * oof_svr_b

print(f'\nPipeline A (freq-encoding, Ridge+GB) RMSE OOF: {rmse(y_a, pipeline_a_oof):,.0f}')
print(f'Pipeline B (one-hot, EN+SVR)          RMSE OOF: {rmse(y_b, pipeline_b_oof):,.0f}')

Fold 1/5 listo
Fold 2/5 listo
Fold 3/5 listo
Fold 4/5 listo
Fold 5/5 listo

Pipeline A (freq-encoding, Ridge+GB) RMSE OOF: 22,538
Pipeline B (one-hot, EN+SVR)          RMSE OOF: 22,241


## 6. Combinar ambos pipelines

In [16]:
def combo_rmse(w):
    combo = w * pipeline_a_oof + (1 - w) * pipeline_b_oof
    return rmse(y_a, combo)

w_grid = np.arange(0, 1.01, 0.01)
best_w = min(w_grid, key=combo_rmse)
print(f'Mejor peso Pipeline A: {best_w:.2f}  (Pipeline B: {1-best_w:.2f})')
print(f'RMSE OOF combinado: {combo_rmse(best_w):,.0f}')

Mejor peso Pipeline A: 0.45  (Pipeline B: 0.55)
RMSE OOF combinado: 21,615


## 7. Corrección de sesgo al des-loguear

**Qué hace esta celda:** corrige un sesgo estadístico conocido como *Duan smearing*. Como el modelo predice el logaritmo del precio y después se revierte con la función exponencial, existe una pequeña subestimación sistemática del precio real. Aquí se busca el multiplicador que minimiza el error al aplicarlo sobre las predicciones combinadas, y se confirma que efectivamente reduce el RMSE (aunque sea una mejora pequeña).

In [17]:
combo_oof = best_w * pipeline_a_oof + (1 - best_w) * pipeline_b_oof
mult_grid = np.arange(0.99, 1.03, 0.001)
best_mult = min(mult_grid, key=lambda m: rmse(y_a, combo_oof * m))
print(f'Mejor multiplicador de corrección: {best_mult:.3f}')
print(f'RMSE OOF sin corrección: {rmse(y_a, combo_oof):,.0f}')
print(f'RMSE OOF con corrección: {rmse(y_a, combo_oof * best_mult):,.0f}')

Mejor multiplicador de corrección: 1.010
RMSE OOF sin corrección: 21,615
RMSE OOF con corrección: 21,535


## 8. Calibración del patrón


In [18]:
def big_qual_partial_flag(df):
    return ((df['GrLivArea'] > 3800) & (df['OverallQual'] >= 9) &
             (df['SaleCondition'] == 'Partial'))

## 9. Entrenamiento final sobre todo el train y predicción sobre el test

In [19]:
# Pipeline A sobre el test
pred_ridge_a_test = fit_ridge_a(Xa, y_a_log, Xa_test)
pred_gb_a_test    = fit_gb_bagged(Xa, y_a_log, Xa_test)
pred_a_test = 0.41 * pred_ridge_a_test + 0.59 * pred_gb_a_test

# Pipeline B sobre el test
pred_en_b_test  = fit_en(Xb, y_b_log, Xb_test)
pred_svr_b_test = fit_svr(Xb, y_b_log, Xb_test)
pred_b_test = 0.145 * pred_en_b_test + 0.855 * pred_svr_b_test

pred_test_naive = (best_w * pred_a_test + (1 - best_w) * pred_b_test) * best_mult

# Predicción "naive" del blend combinado para el ancla (Id 524)
anchor_row_a = add_features_a(train_raw[mask_outlier_a].drop(columns=['Id', 'SalePrice']))
pred_a_anchor = (0.41 * fit_ridge_a(Xa, y_a_log, anchor_row_a) +
                  0.59 * fit_gb_bagged(Xa, y_a_log, anchor_row_a))
pred_b_anchor = (0.145 * fit_en(Xb, y_b_log, Xb_anchor) +
                  0.855 * fit_svr(Xb, y_b_log, Xb_anchor))
pred_anchor_naive = ((best_w * pred_a_anchor + (1 - best_w) * pred_b_anchor) * best_mult)[0]

CALIBRATION_RATIO = anchor_price / pred_anchor_naive
print(f'Predicción naive combinada para el ancla (Id 524): ${pred_anchor_naive:,.0f}')
print(f'Precio real del ancla:                             ${anchor_price:,.0f}')
print(f'Razón de calibración:                              {CALIBRATION_RATIO:.3f}')

flag_test = big_qual_partial_flag(test_raw).values
final_sale_prices = pred_test_naive.copy()
final_sale_prices[flag_test] *= CALIBRATION_RATIO
final_sale_prices = np.minimum(final_sale_prices, y_a.max() * 1.10)
final_sale_prices = np.clip(final_sale_prices, 10_000, None)

print(f'\nFilas del test con el patrón calibrado: {flag_test.sum()}')
if flag_test.sum() > 0:
    print(test_raw.loc[flag_test, ['Id', 'GrLivArea', 'OverallQual', 'SaleCondition']])
    print('Predicción antes de calibrar:', f'${pred_test_naive[flag_test][0]:,.0f}')
    print('Predicción después de calibrar:', f'${final_sale_prices[flag_test][0]:,.0f}')

print(f'\nmin: {final_sale_prices.min():,.0f}  max: {final_sale_prices.max():,.0f}  '
      f'media: {final_sale_prices.mean():,.0f}')

Predicción naive combinada para el ancla (Id 524): $728,102
Precio real del ancla:                             $184,750
Razón de calibración:                              0.254

Filas del test con el patrón calibrado: 1
       Id  GrLivArea  OverallQual SaleCondition
198  1299       5642           10       Partial
Predicción antes de calibrar: $1,140,495
Predicción después de calibrar: $289,391

min: 57,516  max: 465,908  media: 184,072


## 10. Submission

In [20]:
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_sale_prices
})

submission.to_csv('submission.csv', index=False)
print('submission.csv guardado.')
submission.head(10)

submission.csv guardado.


,Id,SalePrice
0,6,162291.919230
1,7,283101.270879
2,8,230332.607846
3,52,117778.213810
4,60,120950.219272
5,65,241444.711883
6,66,297029.008762
7,72,121007.843782
8,79,124764.553474
9,80,117119.537626
